# 10 Personalized Employability Recommendation System

## Objective

The purpose of this notebook is to transform the final machine learning prediction and SHAP explanation into personalized and actionable employability feedback.

The recommendation engine combines:

1. Construct-level Internship Experience scores
2. Local SHAP contributions from the final Tuned Random Forest model
3. Transparent priority rules
4. Construct-specific recommendation mappings

The system identifies student strengths, areas that should be monitored, and priority development areas. Targeted recommendations are generated only when the hybrid analysis identifies a meaningful development need. Students without priority development areas receive maintenance-oriented feedback.

The system retains Employability Score as a continuous prediction rather than converting it into arbitrary High, Medium, or Low employability categories.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import shap

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

d:\internship-employability-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Define project folders

In [2]:
PROJECT_DIR = Path.cwd().parent

DATA_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "descriptive_dataset.csv"
)

MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"
RECOMMENDATION_DIR = REPORT_DIR / "recommendations"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
RECOMMENDATION_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_FILE)

Project: d:\internship-employability-project
Dataset: d:\internship-employability-project\data\processed\descriptive_dataset.csv


In [3]:
df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (1000, 29)


,Gender,Age,Degree_Program,Completed_Internship,SIE1,SIE2,SIE3,SIE4,SIE5,SIE6,...,EMP4,EMP5,EMP6,EMP7,EMP8,Clear_Objectives,School_Support,Workplace_Environment,Career_Development,Employability_Score
0,Male,24–26,Engineering,Yes,4,5,4,4,5,5,...,4,4,4,5,5,4.25,5.0,4.25,4.5,4.625
1,Male,21–23,Information Technology,Yes,5,4,4,4,4,4,...,4,3,5,4,4,4.25,4.0,3.75,3.5,3.875
2,Male,21–23,Information Technology,Yes,4,5,4,4,4,3,...,4,4,5,4,3,4.25,3.5,4.00,4.0,3.875
3,Female,18–20,Engineering,Yes,4,4,4,4,4,3,...,4,4,4,4,4,4.00,3.5,3.75,3.0,4.000
4,Prefer not to say,18–20,Other,Yes,4,5,4,4,4,5,...,4,5,5,4,4,4.25,4.5,4.25,4.5,4.250


Define your final model features

In [4]:
feature_names = [
    "Clear_Objectives",
    "School_Support",
    "Workplace_Environment",
    "Career_Development"
]

target_name = "Employability_Score"

X = df[feature_names].copy()
y = df[target_name].copy()

In [5]:
assert X.isnull().sum().sum() == 0
assert y.isnull().sum() == 0

print("Feature validation passed.")

Feature validation passed.


Recreate the research train/test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (800, 4)
Testing : (200, 4)


Recreate/load the final Tuned Random Forest

In [7]:
final_rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

final_rf_model.fit(
    X_train,
    y_train
)

print("Final model ready.")

Final model ready.


Create the SHAP explainer

In [8]:
explainer = shap.TreeExplainer(
    final_rf_model
)

In [9]:
shap_train = explainer(X_train)
shap_test = explainer(X_test)

print(
    "Training SHAP shape:",
    shap_train.values.shape
)

print(
    "Testing SHAP shape:",
    shap_test.values.shape
)

Training SHAP shape: (800, 4)
Testing SHAP shape: (200, 4)


Calibrate SHAP thresholds using TRAINING data

In [10]:
absolute_training_shap = np.abs(
    shap_train.values
).flatten()

small_effect_threshold = np.quantile(
    absolute_training_shap,
    0.50
)

large_effect_threshold = np.quantile(
    absolute_training_shap,
    0.75
)

print(
    "Small SHAP threshold:",
    round(
        float(small_effect_threshold),
        4
    )
)

print(
    "Large SHAP threshold:",
    round(
        float(large_effect_threshold),
        4
    )
)

Small SHAP threshold: 0.0461
Large SHAP threshold: 0.0863


## SHAP Threshold Calibration

SHAP-magnitude thresholds were calibrated using only the training dataset. This prevents information from the held-out test dataset from influencing the design of the recommendation rules.

The median absolute SHAP value was used to distinguish small effects, while the 75th percentile was used to identify comparatively large effects.

These thresholds are model-relative operational thresholds and are not universal standards.

Define construct-score interpretation

In [11]:
def classify_construct_score(
    score: float
) -> str:

    if score < 3.0:
        return "Low"

    elif score < 4.0:
        return "Moderate"

    return "High"

Define SHAP effect classification

In [12]:
def classify_shap_effect(
    shap_value: float,
    small_threshold: float,
    large_threshold: float
) -> str:

    magnitude = abs(shap_value)

    if magnitude < small_threshold:
        magnitude_label = "Small"

    elif magnitude < large_threshold:
        magnitude_label = "Moderate"

    else:
        magnitude_label = "Large"

    direction = (
        "Positive"
        if shap_value >= 0
        else "Negative"
    )

    return (
        f"{magnitude_label} "
        f"{direction}"
    )

Define hybrid priority logic

In [13]:
def determine_priority(
    score_level: str,
    shap_value: float,
    small_threshold: float,
    large_threshold: float
) -> str:

    magnitude = abs(shap_value)

    # Strong development need
    if (
        score_level == "Low"
        and shap_value < 0
        and magnitude >= large_threshold
    ):
        return "High Priority"

    # Low score with any negative effect
    if (
        score_level == "Low"
        and shap_value < 0
    ):
        return "Priority"

    # Moderate score + meaningful negative effect
    if (
        score_level == "Moderate"
        and shap_value < 0
        and magnitude >= small_threshold
    ):
        return "Moderate Priority"

    # Strong current area
    if (
        score_level == "High"
        and shap_value > 0
    ):
        return "Strength"

    # Emerging positive area
    if (
        score_level == "Moderate"
        and shap_value > 0
    ):
        return "Developing Strength"

    return "Monitor"

Define priority order

In [14]:
priority_order = {
    "High Priority": 1,
    "Priority": 2,
    "Moderate Priority": 3,
    "Monitor": 4,
    "Developing Strength": 5,
    "Strength": 6
}

Create recommendation IDs and mapping

In [15]:
recommendation_map = {

    "Clear_Objectives": [
        {
            "id": "CO_R1",
            "text":
                "Discuss internship goals and expected outcomes with the supervisor."
        },
        {
            "id": "CO_R2",
            "text":
                "Break assigned tasks into clear weekly objectives."
        },
        {
            "id": "CO_R3",
            "text":
                "Request clarification when tasks or performance expectations are unclear."
        }
    ],

    "School_Support": [
        {
            "id": "SS_R1",
            "text":
                "Schedule regular discussions with the academic internship supervisor."
        },
        {
            "id": "SS_R2",
            "text":
                "Use available institutional career and internship support services."
        },
        {
            "id": "SS_R3",
            "text":
                "Request guidance on internship assignments and workplace challenges."
        }
    ],

    "Workplace_Environment": [
        {
            "id": "WE_R1",
            "text":
                "Request regular and constructive feedback from the workplace supervisor."
        },
        {
            "id": "WE_R2",
            "text":
                "Seek opportunities to collaborate with experienced professionals."
        },
        {
            "id": "WE_R3",
            "text":
                "Discuss workplace support needs through the appropriate supervision channel."
        }
    ],

    "Career_Development": [
        {
            "id": "CD_R1",
            "text":
                "Participate in challenging projects that build industry-relevant experience."
        },
        {
            "id": "CD_R2",
            "text":
                "Seek networking, mentoring, and professional development opportunities."
        },
        {
            "id": "CD_R3",
            "text":
                "Document achievements and skills gained during the internship."
        }
    ]
}

Create maintenance feedback

In [16]:
def generate_maintenance_message(
    strengths
):

    if strengths:

        names = [
            item["Construct"].replace(
                "_",
                " "
            )
            for item in strengths
        ]

        strengths_text = ", ".join(
            names
        )

        return (
            "No priority development area was identified. "
            f"Maintain your current strengths in {strengths_text} "
            "and continue seeking opportunities for professional development."
        )

    return (
        "No priority development area was identified. "
        "Continue maintaining your current internship performance "
        "and seek further professional development opportunities."
    )

Create hybrid construct analysis function

In [17]:
def analyse_student_constructs(
    student_input,
    student_shap
):

    analysis = pd.DataFrame({
        "Construct":
            feature_names,

        "Construct Score":
            student_input.iloc[0].values,

        "SHAP Contribution":
            student_shap.values
    })

    analysis["Score Level"] = (
        analysis[
            "Construct Score"
        ].apply(
            classify_construct_score
        )
    )

    analysis["SHAP Effect"] = (
        analysis[
            "SHAP Contribution"
        ].apply(
            lambda value:
                classify_shap_effect(
                    value,
                    small_effect_threshold,
                    large_effect_threshold
                )
        )
    )

    analysis["Priority"] = (
        analysis.apply(
            lambda row:
                determine_priority(
                    row["Score Level"],
                    row[
                        "SHAP Contribution"
                    ],
                    small_effect_threshold,
                    large_effect_threshold
                ),
            axis=1
        )
    )

    analysis["Priority Rank"] = (
        analysis["Priority"]
        .map(priority_order)
    )

    analysis = analysis.sort_values(
        [
            "Priority Rank",
            "SHAP Contribution"
        ],
        ascending=[
            True,
            True
        ]
    )

    return analysis

Generate only the TOP 2 development areas

In [18]:
def generate_recommendations(
    analysis,
    maximum_constructs=2
):

    development_labels = {
        "High Priority",
        "Priority",
        "Moderate Priority"
    }

    priority_rows = (
        analysis[
            analysis["Priority"].isin(
                development_labels
            )
        ]
        .head(
            maximum_constructs
        )
    )

    recommendations = []

    for _, row in priority_rows.iterrows():

        construct = row[
            "Construct"
        ]

        recommendations.append({

            "Construct":
                construct,

            "Construct Score":
                round(
                    float(
                        row[
                            "Construct Score"
                        ]
                    ),
                    2
                ),

            "SHAP Contribution":
                round(
                    float(
                        row[
                            "SHAP Contribution"
                        ]
                    ),
                    4
                ),

            "Priority":
                row["Priority"],

            "Reason":
                (
                    f"{row['Score Level']} construct score "
                    f"with {row['SHAP Effect']} model contribution"
                ),

            "Actions":
                recommendation_map[
                    construct
                ]
        })

    return recommendations

Create a full recommendation report function

In [19]:
def create_recommendation_report(
    student_position: int,
    include_actual=True
):

    if not (
        0 <= student_position < len(X_test)
    ):
        raise IndexError(
            "Student position is outside the test dataset."
        )

    student_input = X_test.iloc[
        [student_position]
    ]

    prediction = float(
        final_rf_model.predict(
            student_input
        )[0]
    )

    student_shap = shap_test[
        student_position
    ]

    analysis = (
        analyse_student_constructs(
            student_input,
            student_shap
        )
    )

    recommendations = (
        generate_recommendations(
            analysis,
            maximum_constructs=2
        )
    )

    strengths_df = (
        analysis[
            analysis[
                "Priority"
            ].isin(
                [
                    "Strength",
                    "Developing Strength"
                ]
            )
        ]
        .sort_values(
            "SHAP Contribution",
            ascending=False
        )
    )

    strengths = strengths_df[
        [
            "Construct",
            "Construct Score",
            "SHAP Contribution"
        ]
    ].to_dict(
        "records"
    )

    if recommendations:

        maintenance_message = None

    else:

        maintenance_message = (
            generate_maintenance_message(
                strengths
            )
        )

    report = {

        "Student Position":
            int(
                student_position
            ),

        "Original Row Index":
            int(
                X_test.index[
                    student_position
                ]
            ),

        "Predicted Employability Score":
            round(
                prediction,
                3
            ),

        "Strengths":
            strengths,

        "Development Recommendations":
            recommendations,

        "Maintenance Message":
            maintenance_message,

        "Full Construct Analysis":
            analysis.drop(
                columns=[
                    "Priority Rank"
                ]
            ).to_dict(
                "records"
            )
    }

    # Research evaluation only
    if include_actual:

        actual = float(
            y_test.iloc[
                student_position
            ]
        )

        report[
            "Actual Employability Score"
        ] = round(
            actual,
            3
        )

        report[
            "Absolute Prediction Error"
        ] = round(
            abs(
                actual
                - prediction
            ),
            3
        )

    return report

In [22]:
# ---------------------------------------------------------
# Select representative low, middle, and high prediction cases
# ---------------------------------------------------------

test_predictions = final_rf_model.predict(
    X_test
)

# Positions sorted from lowest to highest prediction
sorted_positions = np.argsort(
    test_predictions
)

# Three lowest predicted cases
low_positions = (
    sorted_positions[:3]
    .tolist()
)

# Three highest predicted cases
high_positions = (
    sorted_positions[-3:][::-1]
    .tolist()
)

# Find cases closest to the median prediction
median_prediction = np.median(
    test_predictions
)

middle_positions = (
    np.argsort(
        np.abs(
            test_predictions
            - median_prediction
        )
    )[:3]
    .tolist()
)

print(
    "Low positions:",
    low_positions
)

print(
    "Middle positions:",
    middle_positions
)

print(
    "High positions:",
    high_positions
)

Low positions: [194, 52, 77]
Middle positions: [11, 28, 29]
High positions: [132, 102, 81]


In [24]:
selected_case_rows = []

for group_name, positions in [
    ("Low", low_positions),
    ("Middle", middle_positions),
    ("High", high_positions),
]:

    for position in positions:

        selected_case_rows.append({
            "Group":
                group_name,

            "Student Position":
                int(position),

            "Original Row Index":
                int(
                    X_test.index[
                        position
                    ]
                ),

            "Predicted Score":
                round(
                    float(
                        test_predictions[
                            position
                        ]
                    ),
                    3
                ),

            "Actual Score":
                round(
                    float(
                        y_test.iloc[
                            position
                        ]
                    ),
                    3
                )
        })

selected_cases = pd.DataFrame(
    selected_case_rows
)

selected_cases

,Group,Student Position,Original Row Index,Predicted Score,Actual Score
0,Low,194,213,2.812,1.000
1,Low,52,261,3.009,3.375
2,Low,77,39,3.225,3.000
3,Middle,11,76,4.064,4.375
4,Middle,28,902,4.056,4.250
5,Middle,29,947,4.056,4.000
6,High,132,265,4.685,5.000
7,High,102,936,4.685,4.875
8,High,81,66,4.675,5.000


Test low / middle / high cases again

In [25]:
low_report = (
    create_recommendation_report(
        int(low_positions[0])
    )
)

middle_report = (
    create_recommendation_report(
        int(middle_positions[0])
    )
)

high_report = (
    create_recommendation_report(
        int(high_positions[-1])
    )
)

In [26]:
low_report

{'Student Position': 194,
 'Original Row Index': 213,
 'Predicted Employability Score': 2.812,
 'Strengths': [],
 'Development Recommendations': [{'Construct': 'Clear_Objectives',
   'Construct Score': 1.25,
   'SHAP Contribution': -0.5616,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'CO_R1',
     'text': 'Discuss internship goals and expected outcomes with the supervisor.'},
    {'id': 'CO_R2',
     'text': 'Break assigned tasks into clear weekly objectives.'},
    {'id': 'CO_R3',
     'text': 'Request clarification when tasks or performance expectations are unclear.'}]},
  {'Construct': 'Career_Development',
   'Construct Score': 1.0,
   'SHAP Contribution': -0.2987,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'CD_R1',
     'text': 'Participate in challenging projects that build industry-relevant experience.'},
    {

In [27]:
middle_report

{'Student Position': 11,
 'Original Row Index': 76,
 'Predicted Employability Score': 4.064,
 'Strengths': [{'Construct': 'Clear_Objectives',
   'Construct Score': 4.5,
   'SHAP Contribution': 0.18124227386817338},
  {'Construct': 'Career_Development',
   'Construct Score': 4.0,
   'SHAP Contribution': 0.00974465171583741}],
 'Development Recommendations': [{'Construct': 'School_Support',
   'Construct Score': 2.5,
   'SHAP Contribution': -0.1233,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'SS_R1',
     'text': 'Schedule regular discussions with the academic internship supervisor.'},
    {'id': 'SS_R2',
     'text': 'Use available institutional career and internship support services.'},
    {'id': 'SS_R3',
     'text': 'Request guidance on internship assignments and workplace challenges.'}]},
  {'Construct': 'Workplace_Environment',
   'Construct Score': 3.0,
   'SHAP Contribution': -0.1074,
   'Prior

In [28]:
high_report

{'Student Position': 81,
 'Original Row Index': 66,
 'Predicted Employability Score': 4.675,
 'Strengths': [{'Construct': 'Workplace_Environment',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.19798800677574},
  {'Construct': 'School_Support',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.177812326369363},
  {'Construct': 'Clear_Objectives',
   'Construct Score': 4.75,
   'SHAP Contribution': 0.13521933380542203},
  {'Construct': 'Career_Development',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.06047736536999257}],
 'Development Recommendations': [],
 'Maintenance Message': 'No priority development area was identified. Maintain your current strengths in Workplace Environment, School Support, Clear Objectives, Career Development and continue seeking opportunities for professional development.',
 'Full Construct Analysis': [{'Construct': 'Career_Development',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.06047736536999257,
   'Score Level': 'High',
   'SHAP 

Generate reports for ALL test students

In [29]:
all_student_reports = [
    create_recommendation_report(
        i
    )
    for i in range(
        len(X_test)
    )
]

print(
    "Reports generated:",
    len(
        all_student_reports
    )
)

Reports generated: 200


Create a summary dataset

In [30]:
summary_rows = []

for report in all_student_reports:

    recommendations = report[
        "Development Recommendations"
    ]

    strengths = report[
        "Strengths"
    ]

    priority_constructs = [
        item["Construct"]
        for item in recommendations
    ]

    strength_constructs = [
        item["Construct"]
        for item in strengths
    ]

    summary_rows.append({

        "Student Position":
            report[
                "Student Position"
            ],

        "Original Row Index":
            report[
                "Original Row Index"
            ],

        "Predicted Employability Score":
            report[
                "Predicted Employability Score"
            ],

        "Actual Employability Score":
            report[
                "Actual Employability Score"
            ],

        "Absolute Prediction Error":
            report[
                "Absolute Prediction Error"
            ],

        "Strengths":
            ", ".join(
                strength_constructs
            ),

        "Priority Areas":
            ", ".join(
                priority_constructs
            ),

        "Recommendation Count":
            len(
                recommendations
            ),

        "Maintenance Feedback":
            report[
                "Maintenance Message"
            ]
    })

recommendation_summary = pd.DataFrame(
    summary_rows
)

recommendation_summary.head()

,Student Position,Original Row Index,Predicted Employability Score,Actual Employability Score,Absolute Prediction Error,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback
0,0,521,3.986,4.000,0.014,Career_Development,,0,No priority development area was identified. M...
1,1,737,4.338,4.375,0.037,"School_Support, Clear_Objectives, Career_Devel...",,0,No priority development area was identified. M...
2,2,740,4.003,4.000,0.003,Career_Development,,0,No priority development area was identified. M...
3,3,660,4.003,4.000,0.003,Career_Development,,0,No priority development area was identified. M...
4,4,411,4.087,4.500,0.413,"School_Support, Career_Development",,0,No priority development area was identified. M...


Recommendation coverage analysis

In [31]:
total_students = len(
    recommendation_summary
)

students_with_recommendations = (
    recommendation_summary[
        "Recommendation Count"
    ]
    .gt(0)
    .sum()
)

students_without_recommendations = (
    total_students
    - students_with_recommendations
)

coverage_percentage = (
    students_with_recommendations
    / total_students
    * 100
)

print(
    "Total students:",
    total_students
)

print(
    "With targeted recommendations:",
    students_with_recommendations
)

print(
    "With maintenance feedback:",
    students_without_recommendations
)

print(
    "Targeted recommendation coverage:",
    round(
        coverage_percentage,
        2
    ),
    "%"
)

Total students: 200
With targeted recommendations: 34
With maintenance feedback: 166
Targeted recommendation coverage: 17.0 %


Recommendation frequency by construct

In [32]:
construct_counts = {
    feature: 0
    for feature in feature_names
}

for report in all_student_reports:

    for item in report[
        "Development Recommendations"
    ]:

        construct = item[
            "Construct"
        ]

        construct_counts[
            construct
        ] += 1

recommendation_frequency = (
    pd.DataFrame(
        {
            "Construct":
                construct_counts.keys(),

            "Students Recommended":
                construct_counts.values()
        }
    )
    .sort_values(
        "Students Recommended",
        ascending=False
    )
)

recommendation_frequency

,Construct,Students Recommended
0,Clear_Objectives,23
3,Career_Development,19
1,School_Support,7
2,Workplace_Environment,5


Priority distribution

In [33]:
priority_values = []

for report in all_student_reports:

    for item in report[
        "Development Recommendations"
    ]:

        priority_values.append(
            item[
                "Priority"
            ]
        )

priority_distribution = (
    pd.Series(
        priority_values
    )
    .value_counts()
    .rename_axis(
        "Priority"
    )
    .reset_index(
        name="Count"
    )
)

priority_distribution

,Priority,Count
0,Moderate Priority,39
1,High Priority,14
2,Priority,1


Check for empty user-facing outputs

In [34]:
empty_feedback_cases = []

for report in all_student_reports:

    has_recommendation = (
        len(
            report[
                "Development Recommendations"
            ]
        )
        > 0
    )

    has_maintenance = bool(
        report[
            "Maintenance Message"
        ]
    )

    if (
        not has_recommendation
        and not has_maintenance
    ):

        empty_feedback_cases.append(
            report[
                "Student Position"
            ]
        )

print(
    "Empty feedback cases:",
    empty_feedback_cases
)

Empty feedback cases: []


Check low-score recommendation safety

In [35]:
low_prediction_without_advice = (
    recommendation_summary[
        (
            recommendation_summary[
                "Predicted Employability Score"
            ]
            < 3.0
        )
        &
        (
            recommendation_summary[
                "Recommendation Count"
            ]
            == 0
        )
    ]
)

low_prediction_without_advice

,Student Position,Original Row Index,Predicted Employability Score,Actual Employability Score,Absolute Prediction Error,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback


Export the final summary CSV

In [36]:
recommendation_summary.to_csv(
    RECOMMENDATION_DIR
    / "Student_Recommendation_Summary.csv",
    index=False
)

recommendation_frequency.to_csv(
    RECOMMENDATION_DIR
    / "Recommendation_Frequency.csv",
    index=False
)

priority_distribution.to_csv(
    RECOMMENDATION_DIR
    / "Priority_Distribution.csv",
    index=False
)

print(
    "Recommendation summary files saved."
)

Recommendation summary files saved.


Export all detailed reports as JSON

In [37]:
all_reports_file = (
    RECOMMENDATION_DIR
    / "Detailed_Student_Recommendations.json"
)

with open(
    all_reports_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        all_student_reports,
        file,
        indent=4,
        ensure_ascii=False,
        default=float
    )

print(
    "Detailed reports saved."
)

Detailed reports saved.


Create expert-validation dataset

In [38]:
expert_validation_rows = []

for construct, recommendations in (
    recommendation_map.items()
):

    for recommendation in recommendations:

        expert_validation_rows.append({

            "Construct":
                construct,

            "Recommendation ID":
                recommendation[
                    "id"
                ],

            "Recommendation":
                recommendation[
                    "text"
                ],

            "Relevance Rating (1-5)":
                "",

            "Clarity Rating (1-5)":
                "",

            "Practicality Rating (1-5)":
                "",

            "Expert Comments":
                ""
        })

expert_validation_table = (
    pd.DataFrame(
        expert_validation_rows
    )
)

expert_validation_table

,Construct,Recommendation ID,Recommendation,Relevance Rating (1-5),Clarity Rating (1-5),Practicality Rating (1-5),Expert Comments
0,Clear_Objectives,CO_R1,Discuss internship goals and expected outcomes...,,,,
1,Clear_Objectives,CO_R2,Break assigned tasks into clear weekly objecti...,,,,
2,Clear_Objectives,CO_R3,Request clarification when tasks or performanc...,,,,
3,School_Support,SS_R1,Schedule regular discussions with the academic...,,,,
4,School_Support,SS_R2,Use available institutional career and interns...,,,,
5,School_Support,SS_R3,Request guidance on internship assignments and...,,,,
6,Workplace_Environment,WE_R1,Request regular and constructive feedback from...,,,,
7,Workplace_Environment,WE_R2,Seek opportunities to collaborate with experie...,,,,
8,Workplace_Environment,WE_R3,Discuss workplace support needs through the ap...,,,,
9,Career_Development,CD_R1,Participate in challenging projects that build...,,,,


In [39]:
expert_validation_table.to_csv(
    RECOMMENDATION_DIR
    / "Expert_Validation_Template.csv",
    index=False
)

print(
    "Expert validation template saved."
)

Expert validation template saved.


Save recommendation configuration

In [40]:
recommendation_config = {

    "features":
        feature_names,

    "small_shap_threshold":
        float(
            small_effect_threshold
        ),

    "large_shap_threshold":
        float(
            large_effect_threshold
        ),

    "maximum_priority_constructs":
        2,

    "construct_score_rules": {
        "Low": "< 3.0",
        "Moderate": "3.0 to < 4.0",
        "High": ">= 4.0"
    },

    "recommendations":
        recommendation_map,

    "method":
        (
            "Hybrid Construct Score "
            "+ Local SHAP Recommendation System"
        )
}

with open(
    MODEL_DIR
    / "recommendation_system_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        recommendation_config,
        file,
        indent=4,
        ensure_ascii=False
    )

print(
    "Recommendation configuration saved."
)

Recommendation configuration saved.


# Recommendation System Methodology

The personalized recommendation system was developed using a hybrid explainable approach that combines student-level construct scores with local SHAP contributions from the final Tuned Random Forest model.

Construct scores represent the student's current Internship Experience profile, while local SHAP values indicate how each construct influenced the model's predicted Employability Score.

The system classifies each construct as a strength, developing strength, monitoring area, or development priority. Targeted recommendations are generated only for the highest-priority development areas, with a maximum of two constructs selected to avoid recommendation overload.

Students without identified development priorities receive maintenance-oriented feedback that reinforces their current strengths.

The recommendation engine therefore extends the predictive model from a prediction-only system into an explainable decision-support framework consisting of prediction, explanation, prioritization, and actionable feedback.

# Recommendation System Findings

The recommendation engine successfully generated personalized outputs for all held-out test cases without producing empty user-facing reports.

Targeted development recommendations were generated only when the hybrid construct-score and SHAP analysis identified meaningful development needs. Students without priority development areas received maintenance-oriented feedback.

The recommendation-frequency analysis demonstrated that different internship constructs were prioritized across different students, indicating that the system produced individualized rather than uniform recommendations.

The hybrid approach also retained model transparency by recording the construct score, SHAP contribution, priority classification, recommendation reason, and recommendation identifier associated with each action.

These findings demonstrate the feasibility of transforming the employability prediction model into an explainable and personalized decision-support system.

## Limitations

The recommendation rules are prototype operational rules and should not be interpreted as universally validated thresholds.

SHAP identifies how features influence model predictions but does not establish causal relationships.

The current recommendation text has not yet undergone independent domain-expert validation.

The model-development dataset is synthetic; therefore, recommendation behaviour must be evaluated using real participant data before real-world deployment.

Expert review by internship coordinators, career advisers, academic supervisors, or relevant professionals is required before the recommendation mapping is considered validated.

In [41]:
recommendation_summary.head(10)

,Student Position,Original Row Index,Predicted Employability Score,Actual Employability Score,Absolute Prediction Error,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback
0,0,521,3.986,4.000,0.014,Career_Development,,0,No priority development area was identified. M...
1,1,737,4.338,4.375,0.037,"School_Support, Clear_Objectives, Career_Devel...",,0,No priority development area was identified. M...
2,2,740,4.003,4.000,0.003,Career_Development,,0,No priority development area was identified. M...
3,3,660,4.003,4.000,0.003,Career_Development,,0,No priority development area was identified. M...
4,4,411,4.087,4.500,0.413,"School_Support, Career_Development",,0,No priority development area was identified. M...
5,5,678,4.338,4.125,0.213,"School_Support, Clear_Objectives, Career_Devel...",,0,No priority development area was identified. M...
6,6,626,4.317,4.375,0.058,"School_Support, Clear_Objectives",,0,No priority development area was identified. M...
7,7,513,3.986,3.875,0.111,Career_Development,,0,No priority development area was identified. M...
8,8,859,4.352,4.375,0.023,"School_Support, Clear_Objectives, Career_Devel...",,0,No priority development area was identified. M...
9,9,136,3.845,3.625,0.220,Career_Development,Clear_Objectives,1,NaN


In [42]:
recommendation_frequency

,Construct,Students Recommended
0,Clear_Objectives,23
3,Career_Development,19
1,School_Support,7
2,Workplace_Environment,5


In [43]:
priority_distribution

,Priority,Count
0,Moderate Priority,39
1,High Priority,14
2,Priority,1


In [44]:
print(
    "Targeted coverage:",
    coverage_percentage
)

Targeted coverage: 17.0


In [ ]:
empty_feedback_cases

In [45]:
low_prediction_without_advice

,Student Position,Original Row Index,Predicted Employability Score,Actual Employability Score,Absolute Prediction Error,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback


In [46]:
empty_feedback_cases

[]

Expert Validation = Pending